In [2]:
# imports
import json, geopandas as gpd, numpy as np
from pathlib import Path
from scipy.spatial import Delaunay
from shapely.geometry import Polygon

In [ ]:
class Cultivation_Summary1:

    def __init__(self, boundary_dir:Path, count_dir:Path, palm_density, blankspot_dir:Path):

        """ boundary_dir: Path of boundary geojson files,
            count_dir: Path of count geojson files,
            palm_density: planting density of palm field,
            blankspot_dir: Path of blankspot geojson files """
       
        # Store mandatory paths
        self.boundary_path = Path(boundary_dir)
        self.count_path = Path(count_dir)
        self.palm_density = palm_density

        # Handle optional blank directory
        self.blankspot_path = Path(blankspot_dir) if blankspot_dir is not None else None
        
        # create destination to save cultivated output
        self.dest_path = self.count_path.resolve().parent.joinpath('block_summary_outputs')
        self.dest_path.mkdir(parents=True, exist_ok=True)


    def block_properties(self, boundary_path:Path):
        """This function extracts Name, Division and Year of Planting
         if available from the boundary geojson file """
        
        # Load the GeoJSON
        with open(boundary_path, "r") as f:
            geojson_data = json.load(f)

        features = geojson_data.get("features", [])
        results = []

        # Loop through all features
        for feature in features:
            props = feature.get("properties", {})
            
            # Extract only the available fields
            extracted = {key: props[key] for key in ["Code", "Year", "Division"] if key in props}
            
            if extracted:  # Only add if something was found
                results.append(extracted)

        # retruns a list of any block property found 
        return results 


    def boundary_points_pairs(self, bdry_loc: Path, pt_loc: Path, threshold: float = 0.99):
        boundary_points = {}

        # Pre-load point files once
        points_data = {
            pt_file.name: gpd.read_file(pt_file)
            for pt_file in pt_loc.glob("*.geojson")
        }

        # Iterate over boundaries
        for block_boundary in bdry_loc.glob("*.geojson"):
            bdry_gdf = gpd.read_file(block_boundary)
            if bdry_gdf.empty:
                continue

            # Clean geometry
            bdry_gdf = bdry_gdf[bdry_gdf.geometry.notnull()]

            # Ensure valid polygons
            bdry_gdf["geometry"] = bdry_gdf.buffer(0)

            # Iterate over all point datasets
            for pt_name, pt_gdf in points_data.items():
                if pt_gdf.empty:
                    continue

                # Reproject points if needed (transform points, not polygons)
                if pt_gdf.crs != bdry_gdf.crs:
                    pt_gdf = pt_gdf.to_crs(bdry_gdf.crs)

                # Spatial join (efficient with R-tree)
                joined = gpd.sjoin(pt_gdf, bdry_gdf, predicate="within", how="inner")

                # Compute ratio of points inside
                within_ratio = len(joined) / len(pt_gdf)

                if within_ratio >= threshold:
                    boundary_points[block_boundary.name] = pt_name

        return boundary_points
    

    # compute true planting boundary area
    def point_triangulation(self, all_points, max_side_length = 13):

        # Get points geometry and extract x/y coordinates
        ini_gdf_points = all_points[['geometry']]
        points = np.array(list(zip(ini_gdf_points.geometry.x, ini_gdf_points.geometry.y)))

        # Perform Delaunay triangulation
        delaunay_triangles = Delaunay(points)
        triangles, triangle_vertices = [],[]
        
        # Iterate each triangle and filter by side length
        for simplex in delaunay_triangles.simplices:

            # Get the triangle vertices
            triangle_points = points[simplex]
            
            # Calculate side lengths
            side_lengths = [
                np.linalg.norm(triangle_points[0] - triangle_points[1]),
                np.linalg.norm(triangle_points[1] - triangle_points[2]),
                np.linalg.norm(triangle_points[2] - triangle_points[0])
            ]
            
            # If all sides are less than or equal to max_side_length, add to triangles
            if all(length <= max_side_length for length in side_lengths):
                triangles.append(Polygon(triangle_points))
                triangle_vertices.append([tuple(triangle_points[0]), tuple(triangle_points[1]), tuple(triangle_points[2])])

        # Create GeoDataFrame for triangles with vertices info
        gdf_triangles = gpd.GeoDataFrame({
            'geometry': triangles,
            'a': [v[0] for v in triangle_vertices],
            'b': [v[1] for v in triangle_vertices],
            'c': [v[2] for v in triangle_vertices]
        }, crs=ini_gdf_points.crs)

        # Dissolve all triangles into a single polygon
        dissolved_polygon = gdf_triangles.union_all()

        # Compute the area of the dissolved polygon in square meters
        polygon_area_m2 = dissolved_polygon.area

        # Convert area to hectares (1 hectare = 10,000 square meters)
        polygon_area_ha = polygon_area_m2 / 10000

        return polygon_area_ha
    

    # Compute planting density within the boundary
    def compute_plant_density(self, boundary_polygon, polygon_area_ha, points_gdf):

        # Count points within the polygon boundary
        points_within = points_gdf[points_gdf.geometry.within(boundary_polygon)]
        
        # Calculate the average planting density
        density_per_ha = len(points_within) / polygon_area_ha if polygon_area_ha > 0 else 0

        return round(density_per_ha, 2)

    
    
    # execute process
    def process(self):

        # Initialize lists for all_blocks properties
        total_block_area, total_tree_count, total_cultivated_area, total_blankspot_count, total_uncultivated_area = [],[],[],[],[]

        boundary_point_pairs = self.boundary_points_pairs(self.boundary_path, self.count_path)

        # iterate files in boundary and read boundary file
        for boundary_name, point_name in boundary_point_pairs.items():

            # get bounbdary path
            boundary_loc = f'{self.boundary_path}/{boundary_name}'

            # get any block property of Name, Division or Year of planting
            block_properties = self.block_properties(boundary_loc)
            
            # read boundary
            boundary_gdf = gpd.read_file(boundary_loc)

            # Get geometry and compute total area in hectares
            geometry = boundary_gdf.geometry.iloc[0]
            total_area_ha = geometry.area / 10000  # Convert to hectares

            # Append block name and area Ha
            block_name = Path(boundary_name).stem
            block_area = round(total_area_ha, 2)
            total_block_area.append(block_area)
            
            # read QA'd count file and append
            count_points = gpd.read_file(f'{self.count_path}/{point_name}')
            tree_count = len(count_points)
            total_tree_count.append(tree_count)

            # calculate for stand per hectar
            stand_per_ha = round(tree_count / block_area,2)

            if self.palm_density == None:

                # compute the plantable area
                polygon_area_ha = self.point_triangulation(count_points, max_side_length = 13)

                # compute planting density
                density_per_ha = self.compute_plant_density(geometry, polygon_area_ha, count_points)
            
            else:
                density_per_ha = self.palm_density

            # calculate cultivated area and append
            cultivated_area_ha = len(count_points) / density_per_ha if density_per_ha > 0 else 0
            cultivated_area = round(cultivated_area_ha, 2)
            percentage_cultivated_area = round((cultivated_area_ha/total_area_ha)*100,2)
            total_cultivated_area.append(cultivated_area)

            if self.blankspot_path is not None:

                # read QA'd blankspot file and append
                blankspot_points = gpd.read_file(f'{self.blankspot_path}/{point_name}')
                blankspot_count = len(blankspot_points)
                total_blankspot_count.append(blankspot_count)

                # calculate uncultivated area and append
                uncultivated_area_ha = len(blankspot_points) / density_per_ha if density_per_ha > 0 else 0
                uncultivated_area = round(uncultivated_area_ha, 1)
                percentage_uncultivated_area = round((uncultivated_area_ha/total_area_ha)*100,2)
                total_uncultivated_area.append(uncultivated_area)
            
            else:
                blankspot_count = uncultivated_area = 0

            # Create block dictionary
            block_summary_data = {
                'Name': block_name,
                'Block_area (ha)': block_area,
                'Tree_count': tree_count,
                'Stand_per_ha': stand_per_ha,
                'Blankspot_count': blankspot_count,
                'Cultivated_area (ha)': f"{cultivated_area} ({percentage_cultivated_area}%)",
                'Uncultivated_area (ha)': f"{uncultivated_area} ({percentage_uncultivated_area}%)"
            }

            # Update block dictionary with extracted block properties
            block_summary_data.update(block_properties[0])

            # Save as JSON
            self.dest_path
            with open(f"{self.dest_path}/{block_name}.json", "w") as f:
                json.dump(block_summary_data, f, indent=4)   # indent=4 for pretty formatting

            print(f'{block_name}.json a success')

        
        # Create block dictionary
        percentage_total_cultivated = round(sum(total_cultivated_area) / sum(total_block_area)*100,2)
        percentage_total_uncultivated = round(sum(total_uncultivated_area) / sum(total_block_area)*100,2)

        all_block_summary_data = {
            'Block_name': 'all_blocks',
            'Num_of_blocks':len(boundary_point_pairs),
            'Block_area (ha)': sum(total_block_area),
            'Tree_count': sum(total_tree_count),
            'Stand_per_ha':round(sum(total_tree_count)/sum(total_block_area),2),
            'Blankspot_count': sum(total_blankspot_count),
            'Cultivated_area (ha)': f"{round(sum(total_cultivated_area),2)} ({percentage_total_cultivated})%",
            'Uncultivated_area (ha)': f"{round(sum(total_uncultivated_area),2)} ({percentage_total_uncultivated})%"
        }


        # Save as JSON
        self.dest_path
        with open(f"{self.dest_path}/all_blocks.json", "w") as f:
            json.dump(all_block_summary_data, f, indent=4)   # indent=4 for pretty formatting

        print('all_blocks.json a success')


# Paths
boundary_dir = '.../block_boundaries/' # boundary data
count_dir = '.../QA_count/' # post QA'd count
blankspot_dir = '.../QA_blankspot/' #  post QA'd blankspot
planting_density = None #  Planting Density

# Implementation
post_processing = Cultivation_Summary1(boundary_dir, count_dir, planting_density, blankspot_dir)
post_processing.process()

In [ ]:
from pathlib import Path
import geopandas as gpd

def total_polygon_area(folder, crs="EPSG:32630"):
    """
    Compute total area of all polygon GeoJSONs in a folder.

    Parameters:
        folder (str or Path): Directory containing GeoJSON files.
        crs (str): Projected CRS for area calculation (default EPSG:3857).

    Returns:
        float: Total area in hectares.
    """
    folder = Path(folder)
    total_area_m2 = 0.0

    for file in folder.glob("*.geojson"):
        gdf = gpd.read_file(file)

        if gdf.empty:
            continue

        # Reproject to projected CRS for area calculation
        if gdf.crs is None:
            raise ValueError(f"{file.name} has no CRS defined.")
        
        if gdf.crs.to_string() != crs:
            gdf = gdf.to_crs(crs)

        total_area_m2 += gdf.area.sum()

    # Convert to hectares
    total_area_ha = total_area_m2 / 10_000
    return total_area_ha

# Example usage
folder_path = '/Users/../Plot_boundaries'
print("Total area:", total_polygon_area(folder_path), "ha")


Total area: 3561.7485885820442 ha
